In [2]:
%pip install splink
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from splink.linker import DuckDBLinker
from splink.datasets import splink_datasets

os.chdir(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa')

df_l = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa\data\raw\df_l_censo_sin_snsa.parquet')
df_r = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa\data\raw\df_r_rraa_sin_snsa.parquet')

#rraa = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa\data\raw\rraa.parquet')
censo = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa\data\raw\censo.parquet')
extranjeros_vinculados = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa\data\extranjeros_vinculados.parquet')

linker = DuckDBLinker([df_l, df_r])
linker.load_model(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa\output\modelos\modelo4.json')

ModuleNotFoundError: No module named 'splink.linker'

In [ ]:
predictions_max_match_prob_modelo3 = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\output\pred_modelo3.parquet')

In [2]:
df_predictions = linker.predict(threshold_match_probability=0.2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
predictions = df_predictions.as_pandas_dataframe()

In [ ]:
max_match_probabilities = predictions.groupby('unique_id_l')['match_weight'].idxmax()

predictions_max_match_prob = predictions.loc[max_match_probabilities]

from utils.saveFile import guardar_como_parquet

#guardar_como_parquet(predictions_max_match_prob, r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\output\pred_modelo4.parquet')

## Análisis del punto de corte

In [7]:
np.quantile(predictions_max_match_prob["match_probability"], [0.001, 0.01, 0.05, 0.1])

array([0.29146376, 0.95736169, 0.9999608 , 0.9999996 ])

### Corte en mw1 (0.99996)

In [15]:
corte1 = predictions_max_match_prob[predictions_max_match_prob["match_probability"] >= mw1]

corte1.value_counts("gamma_documento")

gamma_documento
 2    2811818
-1      82100
 1       7108
 0        532
Name: count, dtype: int64

In [18]:
corte1[corte1["gamma_documento"] == 0].value_counts("gamma_fecha_nacimiento")

gamma_fecha_nacimiento
2    522
1      8
0      2
Name: count, dtype: int64

### Corte en 0.999

In [7]:
corte2 = predictions_max_match_prob[(predictions_max_match_prob["match_probability"] >= 0.999)]

corte2.value_counts("gamma_documento")

gamma_documento
 3    2833712
-1     126803
 2       8519
 1       1835
 0        597
Name: count, dtype: int64

In [33]:
corte2[corte2["gamma_documento"] == 0].value_counts("gamma_fecha_nacimiento")

gamma_fecha_nacimiento
2    1172
1      15
0       5
Name: count, dtype: int64

In [16]:
corte2_modelo3 = predictions_max_match_prob_modelo3[(predictions_max_match_prob_modelo3["match_probability"] >= 0.999)]

In [18]:
anti_join1 = pd.merge(
    corte2, 
    corte2_modelo3, 
    on='unique_id_l', 
    how='left', 
    indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

In [19]:
anti_join2 = pd.merge(
    corte2_modelo3, 
    corte2, 
    on='unique_id_l', 
    how='left', 
    indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

### Corte en 0.9

In [8]:
corte3 = predictions_max_match_prob[(predictions_max_match_prob["match_probability"] >= 0.9) & (predictions_max_match_prob["match_probability"] < 0.999)]

corte3.value_counts("gamma_documento")

gamma_documento
-1    46411
 3    11700
 0     1126
 2      612
 1      420
Name: count, dtype: int64

In [24]:
corte3[corte3["gamma_documento"] == 0].value_counts("gamma_fecha_nacimiento")

gamma_fecha_nacimiento
 2    864
 1      7
-1      1
Name: count, dtype: int64

In [25]:
records_to_view  = corte3.sample(n=50, random_state = 13).to_dict(orient="records")
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

### Corte en 0.2

In [9]:
corte4 = predictions_max_match_prob[(predictions_max_match_prob["match_probability"] < 0.9)]

corte4.value_counts("gamma_documento")

gamma_documento
-1    20852
 3     3573
 0      998
 1      177
 2       91
Name: count, dtype: int64

In [27]:
corte4[corte4["gamma_documento"] == 0].value_counts("gamma_fecha_nacimiento")

gamma_fecha_nacimiento
2    1579
1      19
0       7
Name: count, dtype: int64

In [28]:
records_to_view  = corte4.sample(n=50, random_state = 13).to_dict(orient="records")
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

In [35]:
corte5 = predictions_max_match_prob[(predictions_max_match_prob["match_probability"] >= 0.8) & (predictions_max_match_prob["match_probability"] < 0.9)]

corte5.value_counts("gamma_documento")

gamma_documento
-1    4211
 2     795
 0     405
 1      19
Name: count, dtype: int64

Reglas para obtener vinculados

In [36]:
id_counts = df_l.rename(columns={"documento":"documento_l"})["documento_l"].value_counts()

corte3["doc_unico"] = corte3["documento_l"].apply(lambda x: 1 if pd.notna(x) and id_counts.get(x, 0) == 1 else 0)

corte3[~pd.isnull(corte3["documento_l"])].value_counts("doc_unico")

C:\Users\lpescetto\AppData\Local\Temp\ipykernel_16616\3133092078.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  corte3["doc_unico"] = corte3["documento_l"].apply(lambda x: 1 if pd.notna(x) and id_counts.get(x, 0) == 1 else 0)


doc_unico
1    12188
0     1670
Name: count, dtype: int64

In [45]:
corte3_ajustado = corte3[((corte3["gamma_documento"] == 3) & (corte3["doc_unico"] == 1)) | 
                         ((corte3["gamma_fecha_nacimiento"] == 3) & (corte3["gamma_primer_nombre"].isin([1,2])) & (corte3["gamma_primer_apellido"].isin([1,2])))]

In [40]:
corte3_resto = corte3[~corte3["unique_id_l"].isin(corte3_ajustado["unique_id_l"])]

In [41]:
corte4["doc_unico"] = corte4["documento_l"].apply(lambda x: 1 if pd.notna(x) and id_counts.get(x, 0) == 1 else 0)

corte4_ajustado = corte4[((corte4["gamma_documento"] == 3) & (corte4["doc_unico"] == 1)) | 
                         ((corte4["gamma_fecha_nacimiento"] == 3) & (corte4["gamma_primer_nombre"].isin([1,2])) & (corte4["gamma_primer_apellido"].isin([1,2])))]

C:\Users\lpescetto\AppData\Local\Temp\ipykernel_16616\1785432461.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  corte4["doc_unico"] = corte4["documento_l"].apply(lambda x: 1 if pd.notna(x) and id_counts.get(x, 0) == 1 else 0)


## No vinculados

In [11]:
no_vinculados = pd.merge(df_l.rename(columns = {"unique_id":"unique_id_l"}), predictions_max_match_prob["unique_id_l"], how = "left", indicator = True)

no_vinculados = no_vinculados[no_vinculados["_merge"] == "left_only"].merge(censo[["id_censo", "tipo_caso"]].rename(columns = {"id_censo":"unique_id_l"}), how = "left").drop(columns="_merge")

In [13]:
no_vinculados.value_counts("tipo_caso")

tipo_caso
capi                          53450
cawiincompletonovarbasicas    13183
cawiasociado                   4203
refugiosmides                  1516
cawisinasociar                 1453
registros                      1275
intemperiemides                 740
capiincompletonovarbasicas      481
recuperacion                    249
refugiossen                     232
intemperiesen                   103
capiincompletovarbasicas        102
cawiincompletovarbasicas         46
Name: count, dtype: int64

In [ ]:
no_vinculados.value_counts("ano_nacimiento")

In [28]:
a = df_r[df_r["documento"] == "11104427"]

b = df_l[df_l["documento"] == "11104427"]

In [15]:
registros = censo[censo["tipo_caso"] == "registros"]

In [12]:
dups = predictions_max_match_prob[predictions_max_match_prob.duplicated(subset="unique_id_r", keep = False)]

In [13]:
records_to_view  = predictions.sample(n=50, random_state = 13).to_dict(orient="records")
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

In [ ]:
linker.comparison_viewer_dashboard(df_predictions, "scv.html", overwrite=True)

# You can view the scv.html file in your browser, or inline in a notbook as follows
from IPython.display import IFrame
IFrame(
    src="./scv.html", width="100%", height=1200
)  

In [20]:
extranjeros = predictions[(predictions["id_pais_documento_l"] != "858") & pd.notnull(predictions["id_pais_documento_l"])]

records_to_view  = extranjeros.sample(n=50, random_state = 13).to_dict(orient="records")
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)